# 第 8 课实验：多 Agent 与子 Agent——架构拆解、通信契约与防冲突实战
### 《现代 AI 与 Agent 系统》Colab 实战实验 · 零配置即开即跑

> **【任务简介】（480 字）**  
> 本实验解决多智能体开发中的三大工程难题：**主上下文被脏日志污染、并发回写状态覆盖踩踏、以及跨 Agent 谎言滚雪球**。  
> 
> 你将基于纯 Python（零第三方库依赖）实现工业级多 Agent 调度内核 \`MinMultiAgent\`：  
> 1. **实现总管-工人子上下文隔离**：编写 \`dispatch_subagent_task\`，为子工人创建独立沙箱上下文，过滤冗余原始日志，仅向总管回传结构化结果；  
> 2. **实现防并发覆盖的 Reducer 合并器**：编写 \`state_merge_reducer\`，在 24 种乱序到达与重试场景下，实现按业务主键去重、时序稳定保序以及幂等去重；  
> 3. **实现独立验证者（Verifier）防谎门禁**：编写 \`verify_worker_claims\`，通过确定性物理单元测试阻断跨 Agent 虚假假说，守护系统一致性。  
> 
> **验收方式**：顺次运行所有 Cell，Cell 5 将自动运行完整的断言测试，输出 100 分验收回执！


In [ ]:
# [Cell 1: MinMultiAgent 内核初始化与消息总线]
import json
import itertools
from typing import List, Dict, Any, Callable

print("[OK] Cell 1 初始化完成！MinMultiAgent 多智能体调度底座已就绪。")


In [ ]:
# [Cell 2: 实验 1 — 总管干净上下文 vs 工人脏上下文隔离实测]

def run_isolated_worker(task: Dict[str, Any]) -> Dict[str, Any]:
    """
    工人在自己独立的脏上下文中排查了 10,000 字的长日志，
    但向总管汇报时，只交回极简的三段式交付包！
    """
    # 模拟工人阅读了 10,000 字的冗长错误栈
    dirty_context_log = "Error stack line... " * 1000  # ~10,000 chars
    
    # 提炼出核心事实
    return {
        "task_id": task["id"],
        "status": "SUCCESS",
        "findings": [f"在 {task['target']} 中定位到并修复了 1 个逻辑缺陷"],
        "summary_chars": 60,
        "dirty_chars": len(dirty_context_log),
        "raw_ref": f"/tmp/runs/{task['id']}.log"
    }

# 模拟总管派发 3 个子任务
tasks = [
    {"id": "task_auth", "target": "auth.py"},
    {"id": "task_db", "target": "database.py"},
    {"id": "task_api", "target": "routes.py"}
]

total_dirty_chars = 0
supervisor_context = []

for t in tasks:
    report = run_isolated_worker(t)
    total_dirty_chars += report["dirty_chars"]
    # 总管只接收纯净事实便签
    supervisor_context.append(report["findings"][0])

total_clean_chars = sum(len(x) for x in supervisor_context)
saved_ratio = round((1 - total_clean_chars / total_dirty_chars) * 100, 1)

print(f"=== 实验 1 实测结果（上下文隔离）===")
print(f"工人处理的总脏日志字符数 : {total_dirty_chars} chars")
print(f"总管最终保留的纯净字符数 : {total_clean_chars} chars")
print(f"💡 上下文空间节省率       : {saved_ratio}% (大局观 100% 保全！)")
assert saved_ratio > 80.0, "上下文隔离必须节省 80% 以上空间"


In [ ]:
# [Cell 3: 实验 2 — 防并发覆盖的 Reducer 合并器实现]

def state_merge_reducer(base_state: Dict[str, Any], patch: Dict[str, Any]) -> Dict[str, Any]:
    """
    【学生任务：实现防并发覆盖的去重保序合并函数】
    业务规则要求:
    1. 幂等性：若 patch['task_id'] 已在 base_state['handled_tasks'] 中，直接返回 base_state（忽略重试）；
    2. 按 id 去重并稳定合并 items 列表（严禁直接覆盖导致前序成果丢失）；
    3. 返回全新字典（纯函数，不可就地修改 base_state）。
    """
    # ========================== TODO: START HERE ==========================
    handled = set(base_state.get("handled_tasks", []))
    if patch.get("task_id") in handled:
        return base_state
        
    new_handled = sorted(list(handled | {patch["task_id"]}))
    
    item_map = {x["id"]: x for x in base_state.get("items", [])}
    for item in patch.get("items", []):
        item_map[item["id"]] = item
        
    merged_items = sorted(list(item_map.values()), key=lambda x: x["id"])
    
    return {
        "items": merged_items,
        "handled_tasks": new_handled
    }
    # =========================== TODO: END HERE ===========================

# 交互自测：测试 4! = 24 种所有可能的到达乱序！
patches = [
    {"task_id": "T1", "items": [{"id": 1, "name": "Apple", "price": 10}]},
    {"task_id": "T2", "items": [{"id": 2, "name": "Banana", "price": 5}]},
    {"task_id": "T2", "items": [{"id": 2, "name": "Banana", "price": 5}]}, # 重试包
    {"task_id": "T3", "items": [{"id": 3, "name": "Cherry", "price": 20}]},
]

results = set()
for p in itertools.permutations(patches):
    state = {"items": [], "handled_tasks": []}
    for delta in p:
        state = state_merge_reducer(state, delta)
    assert len(state["items"]) == 3, "去重后项目必须刚好为 3"
    results.add(tuple(x["id"] for x in state["items"]))

assert len(results) == 1, "所有 24 种乱序下的合并结果必须绝对一致！"
print(">>> [PASS] 实验 2：24 种并发乱序全排列测试 100% 通过！")


In [ ]:
# [Cell 4: 实验 3 — 独立 Verifier 阻断跨 Agent 谎言滚雪球]

def mock_python_test_runner(code_snippet: str) -> bool:
    """理查德·费曼物理最高法官：真实运行 Python 测试"""
    # 如果代码里包含不存在的假模块 fastapi_v4，真实物理报错返回 False
    if "import fastapi_v4" in code_snippet or "fastapi.super_route" in code_snippet:
        return False
    return True

def verify_worker_claims(worker_claim: Dict[str, Any]) -> bool:
    """
    【独立验证者 Verifier】
    不看工人吹嘘的推理散文，只拿物理测试断言检验其提交的代码！
    """
    code = worker_claim.get("submitted_code", "")
    return mock_python_test_runner(code)

# 场景测试：Agent A 编造了一个假模块
worker_a_claim = {
    "conclusion": "我已经基于超强框架 fastapi_v4 重构好了接口！",
    "submitted_code": "import fastapi_v4\n@fastapi.super_route('/login')\ndef login(): pass"
}

is_verified = verify_worker_claims(worker_a_claim)
assert is_verified is False, "虚假假说必须被独立 Verifier 一票否决！"
print(">>> [PASS] 实验 3：跨 Agent 谎言被独立物理验证者精准捕获并拦截！")


In [ ]:
# [Cell 5: 自动验收与课程回执生成（Grading Receipt）]
receipt = {
    "lesson": "lesson-08-multi-agent-collaboration",
    "status": "PASSED",
    "metrics": {
        "exp1_clean_context_ratio_saved_pct": saved_ratio,
        "exp2_reducer_permutations_passed": len(list(itertools.permutations(patches))),
        "exp2_duplicate_dedup_success": True,
        "exp3_hallucination_blocked_count": 1,
        "exp3_verifier_accuracy_pct": 100.0
    }
}

print("🎉 恭喜！第 8 课全部 3 个实验验收通过！成绩回执如下：")
print(json.dumps(receipt, indent=2, ensure_ascii=False))
